07 — Preparação dos dados (ADHD-200)
Objetivo: transformar as matrizes de conectividade funcional em atributos para os modelos de classificação.
Etapas:
1. carregar a tabela validada de participantes e as matrizes de conectividade;
2. verificar IDs, rótulos, dimensões e valores inválidos;
3. aplicar a transformação de Fisher z às correlações;
4. vetorizar o triângulo superior sem a diagonal: 190 × 189 / 2 = 17.955 atributos por participante;
5. salvar um único conjunto de dados preparado.
> Decisão metodológica importante: este notebook não cria uma divisão fixa treino/teste. Como a amostra é multicêntrica e há sites com apenas participantes controle, uma divisão aleatória pode causar vazamento de informação relacionada ao centro e dificultar a avaliação realista. A validação e as divisões devem ser definidas nos notebooks de modelagem (08 e 09), respeitando os grupos por site sempre que a estratégia for viável. Qualquer padronização ou seleção de atributos aprendida a partir dos dados deve ser ajustada somente dentro de cada fold de treino.

In [18]:

from pathlib import Path
import sys
import numpy as np
import pandas as pd

# ============================================================
# 1. LOCALIZAR E IMPORTAR O CONFIG.PY DO PROJETO
# ============================================================

diretorio_atual = Path.cwd().resolve()
candidatos = [diretorio_atual, *diretorio_atual.parents]

raiz_projeto = next(
    (
        pasta for pasta in candidatos
        if (pasta / "config.py").is_file()
        and (pasta / "notebooks").is_dir()
    ),
    None
)

if raiz_projeto is None:
    raise FileNotFoundError(
        "Não encontrei a raiz do projeto com config.py e notebooks. "
        f"Diretório atual: {diretorio_atual}"
    )

if str(raiz_projeto) not in sys.path:
    sys.path.insert(0, str(raiz_projeto))

import config

print("Raiz do projeto:", raiz_projeto)
print("Config utilizado:", config.__file__)

# ============================================================
# 2. CONFERIR ARQUIVOS E PASTAS
# ============================================================

config.criar_pastas()

if not config.ARQ_PARTICIPANTES.is_file():
    raise FileNotFoundError(
        f"Arquivo de participantes não encontrado: "
        f"{config.ARQ_PARTICIPANTES}"
    )

if not config.CONNECTIVITY.is_dir():
    raise FileNotFoundError(
        f"Pasta de conectividade não encontrada: "
        f"{config.CONNECTIVITY}"
    )

# ============================================================
# 3. CARREGAR PARTICIPANTES
# ============================================================

df = pd.read_csv(
    config.ARQ_PARTICIPANTES,
    dtype={config.ID_COLUMN: str}
)

colunas_necessarias = [
    config.ID_COLUMN,
    config.TARGET_COLUMN,
    config.SITE_COLUMN
]

ausentes = [
    coluna for coluna in colunas_necessarias
    if coluna not in df.columns
]

if ausentes:
    raise ValueError(
        f"Colunas ausentes em {config.ARQ_PARTICIPANTES.name}: "
        f"{ausentes}. Colunas encontradas: {df.columns.tolist()}"
    )

if df.empty:
    raise ValueError("O arquivo de participantes está vazio.")

if df[config.ID_COLUMN].isna().any():
    raise ValueError("Existem participantes sem identificador.")

if df[config.ID_COLUMN].duplicated().any():
    raise ValueError("Existem identificadores duplicados.")

if df[config.TARGET_COLUMN].isna().any():
    raise ValueError("Existem participantes sem classe.")

classes = set(df[config.TARGET_COLUMN].unique())

classes_esperadas = {
    config.CONTROL_LABEL,
    config.ADHD_LABEL
}

if classes != classes_esperadas:
    raise ValueError(
        f"Classes encontradas: {sorted(classes)}. "
        f"Esperadas: {sorted(classes_esperadas)}. "
        "Confira a codificação de participants.csv."
    )

ids = df[config.ID_COLUMN].astype(str).to_numpy()
y = df[config.TARGET_COLUMN].to_numpy(dtype=np.int64)
sites = df[config.SITE_COLUMN].to_numpy()

# ============================================================
# 4. CONFERIR E CARREGAR AS MATRIZES FC
# ============================================================

fc_ausentes = [
    str(config.CONNECTIVITY / f"FC_{pid}.npy")
    for pid in ids
    if not (config.CONNECTIVITY / f"FC_{pid}.npy").is_file()
]

if fc_ausentes:
    raise FileNotFoundError(
        f"Faltam {len(fc_ausentes)} matrizes de conectividade. "
        f"Exemplos: {fc_ausentes[:5]}"
    )

fc = config.carregar_fc(ids)

formato_esperado = (
    len(df),
    config.N_ROIS,
    config.N_ROIS
)

if fc.shape != formato_esperado:
    raise ValueError(
        f"Formato de fc: {fc.shape}; esperado: {formato_esperado}."
    )

if not np.isfinite(fc).all():
    raise ValueError("As matrizes FC contêm NaN ou infinitos.")

if np.any(fc < -1.000001) or np.any(fc > 1.000001):
    raise ValueError("Existem valores FC fora do intervalo [-1, 1].")

# ============================================================
# 5. RESUMO
# ============================================================

print("\nParticipantes:", len(df))
print("Formato das matrizes FC:", fc.shape)
print("Número de ROIs:", config.N_ROIS)
print("Número de atributos esperados:", config.N_CONNECTIONS)
print("\nDistribuição das classes:")
print(df[config.TARGET_COLUMN].value_counts().sort_index())
print("\nNúmero de sites:", df[config.SITE_COLUMN].nunique())
print("\nCélula 1 concluída.")


Raiz do projeto: C:\Users\Maria Eduarda\Downloads\TCC_TDAH
Config utilizado: C:\Users\Maria Eduarda\Downloads\TCC_TDAH\config.py


ValueError: Colunas ausentes em participants.csv: ['participant_id', 'target']. Colunas encontradas: ['site', 'subject_id', 'arquivo', 'caminho', 'ID_padronizado', 'ScanDir ID', 'Site', 'Gender', 'Age', 'Handedness', 'DX', 'Secondary Dx', 'ADHD Measure', 'ADHD Index', 'Inattentive', 'Hyper/Impulsive', 'IQ Measure', 'Verbal IQ', 'Performance IQ', 'Full2 IQ', 'Full4 IQ', 'Med Status', 'QC_Athena', 'QC_NIAK', 'classe']